# Step 5 — Feature Engineering & Target Construction

**Learning progress**

| Step | Topic | Status |
|------|-------|--------|
| 0 | Problem formulation | COMPLETE |
| 1 | Transaction-level EDA | COMPLETE |
| 2 | Transaction data cleaning | COMPLETE |
| 3 | Clean-data validation | COMPLETE |
| 4 | Customer-level EDA | COMPLETE |
| **5** | **Feature engineering + target construction** | **this notebook** |

---

## Section 1 — Where We Are

### The pipeline so far

```
Raw Excel file  (online_retail_II.xlsx)
    ↓  Step 1: Explore  — understand products, dates, volumes
    ↓  Step 2: Clean    — remove returns, missing IDs, system codes
    ↓  Step 3: Validate — assert every cleaning rule held
    ↓  Step 4: Customer EDA — understand customer behaviour
    ↓
data/interim/transactions_cleaned.csv
    ↓  Step 5 (THIS notebook): Feature engineering
    ↓
data/processed/customer_dataset.csv   ← modelling-ready
    ↓  Step 6: Regression modelling (OLS, Ridge, Lasso, ElasticNet)
```

### Why this step is the bridge

After cleaning we have **84,955 transaction rows** — one row per product line item.
A linear regression model needs **one row per customer** with a fixed set of numbers.

> How do we turn a customer's purchase history (variable-length sequence of transactions)
> into a **fixed-width numerical row** that a regression can learn from?

The answer: build **summary statistics** over each customer's history.
- These summaries are the **features X**
- The customer's future spending is the **target y**

**The most important constraint:** X must come from the PAST; y must come from the FUTURE.
They must never overlap — otherwise training the model is cheating.

### What this notebook teaches

For every feature you will see exactly:
- **CONCEPT** — what it means in business terms
- **FORMULA** — the precise mathematical definition
- **MANUAL PANDAS** — step-by-step implementation you could reproduce alone
- **INSPECT** — what the distribution looks like
- **EDGE CASES** — what can go wrong
- **PRODUCTION FUNCTION** — the reusable `src/` code
- **VERIFY** — assert both approaches produce identical results

---

## Section 2 — Setup and Load Clean Data

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import src as _src_pkg
if 'ecommerce_customer_value_prediction' not in sys.modules:
    sys.modules['ecommerce_customer_value_prediction'] = _src_pkg

import warnings
warnings.filterwarnings('ignore')

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)

from ecommerce_customer_value_prediction.config import INTERIM_DATA_DIR, PROCESSED_DATA_DIR
from ecommerce_customer_value_prediction.features.build_features import (
    PREDICTION_DATE, CUSTOMER_DATASET_CSV,
    build_rfm_features, build_target, build_customer_dataset, save_customer_dataset,
)

CLEANED_CSV = INTERIM_DATA_DIR / 'transactions_cleaned.csv'
print(f'Libraries loaded.')
print(f'Prediction date (from project config): {PREDICTION_DATE.date()}')

In [ ]:
df = pd.read_csv(CLEANED_CSV, parse_dates=['InvoiceDate'])
df['Customer ID'] = df['Customer ID'].astype('Int64')
df['Invoice']     = df['Invoice'].astype(str)
df['StockCode']   = df['StockCode'].astype(str)

print(f'Cleaned dataset loaded.')
print(f'  Shape    : {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'  Columns  : {list(df.columns)}')
print(f'  Dates    : {df["InvoiceDate"].min().date()} to {df["InvoiceDate"].max().date()}')

In [ ]:
df.head(6)

In [ ]:
df.dtypes

### What does one row currently represent?

Each row is **one product line item** inside one invoice.
If a customer bought 5 different products in one trip, that creates 5 rows — all with the same `Invoice` ID.

### Why can we NOT give this to Linear Regression directly?

Linear Regression needs a **fixed-width input matrix** where:
- Each row = one observation (prediction unit)
- Each column = one numeric feature

Our prediction unit is the **customer**.
One customer can occupy anywhere from 1 to 2,000+ rows — variable length sequences cannot
be fed directly into a regression model.

We must **collapse** each customer's entire history into exactly one row of summary statistics.

In [ ]:
rows_per_customer = df.groupby('Customer ID').size()
print('Rows per customer:')
print(f'  Min    : {rows_per_customer.min()}')
print(f'  Median : {rows_per_customer.median():.0f}')
print(f'  Mean   : {rows_per_customer.mean():.1f}')
print(f'  Max    : {rows_per_customer.max()}')
print()
print(f'Current : {len(df):,} rows')
print(f'Target  : {rows_per_customer.shape[0]:,} rows (one per customer)')

---

## Section 3 — Temporal Split: X from the Past, y from the Future

### The prediction date

We simulate a real deployment by choosing a **prediction date** (cutoff = `2011-09-01`).
Imagine the model is asked on that date:

> "Given what this customer has done up to today, how much will they spend in the next few months?"

On that day the future has not happened yet.
Features can only use data **before** that date.
The target uses data **on or after** that date.

```
PAST                          PREDICTION DATE                      FUTURE
|-----------------------------------|---------------------------------|-->
Dec 2009 .............. Aug 2011   |  Sep 2011 ............. Dec 2011
Historical customer behaviour       |  Future customer revenue
Source of X  (features)             |  Source of y  (target)
                                    |
                               2011-09-01
```

### Temporal leakage — WRONG vs CORRECT

**WRONG (data leakage):**
```python
# monetary includes post-cutoff spend — the future is visible in X!
monetary = df.groupby('Customer ID')['line_total'].sum()
```
If future spending bleeds into a feature, the model sees the answer (y)
encoded inside its own input (X). It appears accurate in validation but
fails completely in production where the future does not exist yet.

**CORRECT:**
```python
pre = df[df['InvoiceDate'] < PREDICTION_DATE]   # historical only
monetary = pre.groupby('Customer ID')['line_total'].sum()
```

In [ ]:
CUTOFF = PREDICTION_DATE   # 2011-09-01

historical_df = df[df['InvoiceDate'] < CUTOFF].copy()
future_df     = df[df['InvoiceDate'] >= CUTOFF].copy()

# Verify the two windows are truly non-overlapping
assert (historical_df['InvoiceDate'] < CUTOFF).all(),  'BUG: historical contains post-cutoff rows'
assert (future_df['InvoiceDate']    >= CUTOFF).all(),  'BUG: future contains pre-cutoff rows'

hist_days = (historical_df['InvoiceDate'].max() - historical_df['InvoiceDate'].min()).days
fut_days  = (future_df['InvoiceDate'].max()    - future_df['InvoiceDate'].min()).days

print(f'Cutoff            : {CUTOFF.date()}')
print()
print('HISTORICAL WINDOW — source of X:')
print(f'  Dates    : {historical_df["InvoiceDate"].min().date()} to {historical_df["InvoiceDate"].max().date()}')
print(f'  Length   : {hist_days} days  (~{hist_days/30:.1f} months)')
print(f'  Rows     : {len(historical_df):,}')
print(f'  Customers: {historical_df["Customer ID"].nunique():,}')
print()
print('FUTURE WINDOW — source of y:')
print(f'  Dates    : {future_df["InvoiceDate"].min().date()} to {future_df["InvoiceDate"].max().date()}')
print(f'  Length   : {fut_days} days  (~{fut_days/30:.1f} months)')
print(f'  Rows     : {len(future_df):,}')
print(f'  Customers: {future_df["Customer ID"].nunique():,}')

### Forecast horizon confirmed: 99 days (~3.3 months)

The dataset ends on 2011-12-09, giving a future window of 99 days from the cutoff.
Always verify the actual data range rather than assuming "4 months" or "3 months".

**Rule for the rest of this notebook:**
- Every feature calculation: `historical_df` only
- Target calculation: `future_df` only
- These two DataFrames are never merged or mixed during calculation

---

## Section 4 — Manual Customer Example

Before building any features, let's inspect one real customer's complete history.
This makes the transformation from "many rows → one row" visually concrete.

In [ ]:
# Customer 12374 has both historical and future transactions — ideal for teaching
EXAMPLE_CID = 12374

h_txns = historical_df[historical_df['Customer ID'] == EXAMPLE_CID][
    ['Invoice', 'InvoiceDate', 'StockCode', 'Description', 'Quantity', 'Price', 'line_total', 'Country']
].sort_values('InvoiceDate')

f_txns = future_df[future_df['Customer ID'] == EXAMPLE_CID][
    ['Invoice', 'InvoiceDate', 'StockCode', 'Description', 'Quantity', 'Price', 'line_total']
].sort_values('InvoiceDate')

print(f'Customer {EXAMPLE_CID} — HISTORICAL ({len(h_txns)} rows, {h_txns["Invoice"].nunique()} invoices):')
print(h_txns.to_string(index=False))
print()
print(f'Customer {EXAMPLE_CID} — FUTURE ({len(f_txns)} rows):')
print(f_txns.to_string(index=False))

### Key observation

This customer has **7 historical rows** but only **2 purchase events** (invoices 506579 and 526941).
We need to turn these 7 rows into exactly **one feature row**.

| Feature to build | What to look for |
|---|---|
| `recency_days` | Distance from last purchase to cutoff |
| `frequency` | Number of DISTINCT invoices (= 2, not 7) |
| `monetary` | Sum of all line_total values |
| `avg_basket_value` | Mean spend per invoice |
| `unique_products` | Number of distinct StockCodes |
| `tenure_days` | Gap between first and last purchase |
| `purchase_rate` | frequency / tenure_days |
| `country` | Country from most recent invoice |

In [ ]:
# Quick preview of what the feature row should look like
print('Distinct invoices :', h_txns['Invoice'].nunique())
print('Transaction rows  :', len(h_txns))
print('Distinct products :', h_txns['StockCode'].nunique())
print('Total spend       : £', h_txns['line_total'].sum())
print('First purchase    :', h_txns['InvoiceDate'].min().date())
print('Last purchase     :', h_txns['InvoiceDate'].max().date())
print()
print('Future spend      : £', f_txns['line_total'].sum())

---

## Section 5 — Recency

### CONCEPT

**Recency** answers: "How long ago did this customer last buy?"
It is the 'R' in the classic **RFM** framework.

**Business intuition:**
A customer who bought 3 days ago is clearly still engaged.
A customer who last bought 600 days ago has probably moved on.
Recency is often the single strongest predictor of future purchasing.

### FORMULA

```
recency_days = prediction_date − max(InvoiceDate)  [using historical window only]
```

Lower = more recently active = more likely to return.

In [ ]:
# STEP 1 — find the most recent historical purchase for each customer
last_purchase = (
    historical_df
    .groupby('Customer ID')['InvoiceDate']
    .max()
    .rename('last_purchase')
)

print('Last historical purchase per customer (first 5):')
print(last_purchase.head())

In [ ]:
# STEP 2 — subtract from prediction date to get days since last purchase
# (CUTOFF - datetime) produces a Timedelta; .dt.days gives the integer days
recency = (CUTOFF - last_purchase).dt.days.rename('recency_days')

print('recency_days (first 5):')
print(recency.head())

**What is `.dt.days`?**
Subtracting two datetime objects in pandas produces a `Timedelta` object, e.g. `Timedelta('30 days')`.
The `.dt.days` accessor extracts the integer day count from that Timedelta.
Without it you'd get `Timedelta('30 days 00:00:00')` — not a number a model can use.

In [ ]:
# INSPECT — distribution
print('Recency distribution (days since last historical purchase):')
for label, val in [('Min', recency.min()), ('25th %', recency.quantile(0.25)),
                   ('Median', recency.median()), ('Mean', recency.mean()),
                   ('75th %', recency.quantile(0.75)), ('Max', recency.max())]:
    print(f'  {label:<8}: {val:.1f}')
print(f'  Skew    : {recency.skew():.2f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(recency, bins=50, color='mediumseagreen', edgecolor='white')
ax.axvline(recency.median(), color='black', linestyle='--', label=f'Median={recency.median():.0f}d')
ax.set_xlabel('Recency (days)')
ax.set_ylabel('Number of customers')
ax.set_title('Recency Distribution')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# VERIFY example customer manually
ex_recency = recency.loc[EXAMPLE_CID]
manual     = (CUTOFF - h_txns['InvoiceDate'].max()).days
print(f'Customer {EXAMPLE_CID}  recency_days = {ex_recency}')
print(f'Manual check: {CUTOFF.date()} - {h_txns["InvoiceDate"].max().date()} = {manual} days')
assert ex_recency == manual

### Edge cases
- `recency_days = 0` — customer purchased on the exact prediction date. Valid.
- `recency_days = 638` — customer purchased once in Dec 2009 and never returned. Valid.
- Recency is always ≥ 0 because we only ever look at dates before the cutoff.

---

## Section 6 — Frequency

### CONCEPT

**Frequency** counts how many distinct shopping trips a customer made historically.
It is the 'F' in RFM.

### FORMULA

```
frequency = COUNT(DISTINCT Invoice)  [historical window only]
```

### Critical distinction: invoice count vs. row count

One shopping trip = one Invoice number.
One invoice can contain many products (many line-item rows).
**Row count overestimates purchasing frequency** for customers who buy many products per visit.

In [ ]:
# Demonstrate the difference for our example customer
print(f'Customer {EXAMPLE_CID}:')
print(f'  Transaction rows          : {len(h_txns)}')
print(f'  Distinct invoices (trips) : {h_txns["Invoice"].nunique()}')
print()
print('Rows per invoice:')
print(h_txns.groupby('Invoice').size().rename('line_items').to_string())
print()
print('=> frequency = 2  (not 7)')

In [ ]:
# MANUAL PANDAS — .nunique() counts distinct values within each group
frequency = (
    historical_df
    .groupby('Customer ID')['Invoice']
    .nunique()
    .rename('frequency')
)

print('frequency (first 5):')
print(frequency.head())

In [ ]:
# INSPECT
print('Frequency distribution:')
for label, val in [('Min', frequency.min()), ('Median', frequency.median()),
                   ('Mean', frequency.mean()), ('Max', frequency.max())]:
    print(f'  {label:<8}: {val:.1f}')
print(f'  Skew    : {frequency.skew():.2f}  <- strongly right-skewed')
print(f'  freq=1  : {(frequency==1).sum():,} ({(frequency==1).mean()*100:.1f}%)')
print(f'  freq>50 : {(frequency>50).sum():,}')

cap = frequency.quantile(0.95)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(frequency[frequency <= cap], bins=40, color='steelblue', edgecolor='white')
ax.set_xlabel('Frequency (distinct invoices)')
ax.set_ylabel('Customers')
ax.set_title(f'Frequency Distribution (capped at 95th pct = {cap:.0f})')
plt.tight_layout()
plt.show()

In [ ]:
# VERIFY example customer
ex_freq = frequency.loc[EXAMPLE_CID]
manual  = h_txns['Invoice'].nunique()
print(f'Customer {EXAMPLE_CID}  frequency = {ex_freq}  (manual = {manual})')
assert ex_freq == manual

### Why could frequency predict future value?

From EDA (Step 4): returning customers had **higher** historical frequency (median ~5) than
non-returning customers (median ~1). A customer who has ordered 20 times is more likely to
order again than one who ordered once 2 years ago.

### Multicollinearity note
Frequency correlates strongly with `monetary` (r ≈ +0.56) and `unique_products` (r ≈ +0.75).
This is expected — frequent buyers also spend more and buy more product variety.
It does not mean we should drop these features. It means Ridge/Lasso will be valuable in Step 6.

---

## Section 7 — Monetary (Historical)

### CONCEPT

**Monetary** = total historical spend per customer.
It is the 'M' in RFM.

**Important: historical monetary ≠ future revenue.**
- `monetary` = how much they spent BEFORE the cutoff (a feature, X)
- `future_revenue` = how much they spent AFTER the cutoff (the target, y)

These are different variables measured in different time windows.

### FORMULA

```
monetary = SUM(line_total)  [historical window only]
```

In [ ]:
monetary = (
    historical_df
    .groupby('Customer ID')['line_total']
    .sum()
    .rename('monetary')
)

print('monetary (first 5):')
print(monetary.head())

In [ ]:
# INSPECT — extreme right skew is typical in retail
print('Monetary distribution (historical spend per customer):')
for label, val in [('Min', monetary.min()), ('10th %', monetary.quantile(0.10)),
                   ('Median', monetary.median()), ('Mean', monetary.mean()),
                   ('90th %', monetary.quantile(0.90)), ('99th %', monetary.quantile(0.99)),
                   ('Max', monetary.max())]:
    print(f'  {label:<8}: £{val:>12,.2f}')
print(f'  Skew    : {monetary.skew():.2f}')
print(f'  Mean/Median ratio: {monetary.mean()/monetary.median():.1f}x  (> 1 = right skew)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cap = monetary.quantile(0.99)
axes[0].hist(monetary[monetary <= cap], bins=50, color='darkorange', edgecolor='white')
axes[0].set_xlabel('Historical Monetary Value (£)')
axes[0].set_ylabel('Customers')
axes[0].set_title('Monetary — raw (capped at 99th pct)')
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'£{x:,.0f}'))

axes[1].hist(np.log1p(monetary), bins=50, color='darkorange', edgecolor='white')
axes[1].set_xlabel('log1p(Monetary)')
axes[1].set_ylabel('Customers')
axes[1].set_title('Monetary — log1p scale')

plt.tight_layout()
plt.show()

### Why is monetary right-skewed?

Retail data often follows the **Pareto principle**: roughly 20% of customers generate 80% of revenue.
A small number of business/wholesale buyers can spend 100× more than typical customers.
This creates a very long right tail — most customers cluster near the median (£98),
but a handful can spend £10,000–£60,000.

The log1p distribution (right chart) is approximately bell-shaped — this is why
`log1p` transformations are useful both for features and the target.

In [ ]:
# VERIFY example customer
ex_mon = monetary.loc[EXAMPLE_CID]
manual = h_txns['line_total'].sum()
print(f'Customer {EXAMPLE_CID}  monetary = £{ex_mon:.2f}  (manual = £{manual:.2f})')
assert abs(ex_mon - manual) < 0.01

---

## Section 8 — Average Basket Value (avg_basket_value)

### CONCEPT

**Average basket value** (also called average order value, AOV) measures how much
a customer spends **per shopping trip on average**.

### Why not just use monetary?

Two customers can have identical `monetary` but very different purchasing styles:

| Customer | frequency | avg_basket_value | monetary |
|---|---|---|---|
| A — regular small orders | 20 | £50 | £1,000 |
| B — rare large orders | 2 | £500 | £1,000 |

Both spent £1,000 historically. But their future behaviour may differ:
- Customer A is a habitual small-basket buyer — likely to return regularly
- Customer B is an occasional big spender — less predictable

`avg_basket_value` separates **how often** from **how much per trip**.

### Formula

```
avg_basket_value = mean( sum(line_total per invoice) )
```

Note: we must first aggregate to invoice level (sum line items within each invoice),
then take the mean across invoices. This avoids giving more weight to invoices with
more line items.

In [ ]:
# STEP 1 — sum line_total within each (Customer, Invoice) pair
invoice_spend = (
    historical_df
    .groupby(['Customer ID', 'Invoice'])['line_total']
    .sum()
    .reset_index()
    .rename(columns={'line_total': 'invoice_spend'})
)

print('Invoice-level spend (first 5 rows):')
print(invoice_spend.head())

In [ ]:
# STEP 2 — average across invoices for each customer
avg_basket_value = (
    invoice_spend
    .groupby('Customer ID')['invoice_spend']
    .mean()
    .rename('avg_basket_value')
)

print('avg_basket_value (first 5):')
print(avg_basket_value.head())

In [ ]:
# INSPECT
print('Average Basket Value distribution:')
for label, val in [('Min', avg_basket_value.min()), ('Median', avg_basket_value.median()),
                   ('Mean', avg_basket_value.mean()), ('99th %', avg_basket_value.quantile(0.99)),
                   ('Max', avg_basket_value.max())]:
    print(f'  {label:<8}: £{val:>10,.2f}')
print(f'  Skew    : {avg_basket_value.skew():.2f}')

In [ ]:
# VERIFY: for a single-invoice customer, avg_basket_value == monetary
single_inv = frequency[frequency == 1].index
check = (avg_basket_value.loc[single_inv] - monetary.loc[single_inv]).abs().max()
print(f'Max difference between avg_basket_value and monetary for single-invoice customers: {check:.6f}')
assert check < 1e-6, 'Single-invoice customers must have avg_basket_value == monetary'
print('Confirmed: avg_basket_value == monetary when frequency == 1')

In [ ]:
# VERIFY example customer
inv_spends_ex = invoice_spend[invoice_spend['Customer ID'] == EXAMPLE_CID]['invoice_spend']
print(f'Invoice-level spends for customer {EXAMPLE_CID}:')
print(inv_spends_ex.values)
print(f'Mean: £{inv_spends_ex.mean():.3f}')
print(f'avg_basket_value from our Series: £{avg_basket_value.loc[EXAMPLE_CID]:.3f}')

### Multicollinearity alert: monetary = frequency × avg_basket_value

This is an **exact algebraic identity** (approximately):
```
total_spend ≈ number_of_trips × avg_spend_per_trip
```

Including all three (`monetary`, `frequency`, `avg_basket_value`) in OLS regression creates
near-perfect multicollinearity. OLS will produce large, unstable coefficients.
Ridge, Lasso, and ElasticNet regularization handles this much better — which is exactly why
we study all four in Step 6.

---

## Section 9 — Unique Products

### CONCEPT

**Unique products** counts how many different `StockCode` values a customer purchased historically.

It represents:
- **Breadth of engagement** — buying 50 different products vs. buying the same product 50 times
- **Customer loyalty depth** — customers who explore more product categories may be more engaged

### Formula

```
unique_products = COUNT(DISTINCT StockCode)  [historical window only]
```

In [ ]:
unique_products = (
    historical_df
    .groupby('Customer ID')['StockCode']
    .nunique()
    .rename('unique_products')
)

print('unique_products (first 5):')
print(unique_products.head())

In [ ]:
# INSPECT
print('Unique products distribution:')
for label, val in [('Min', unique_products.min()), ('Median', unique_products.median()),
                   ('Mean', unique_products.mean()), ('99th %', unique_products.quantile(0.99)),
                   ('Max', unique_products.max())]:
    print(f'  {label:<8}: {val:.1f}')
print(f'  Skew    : {unique_products.skew():.2f}')

# Correlation with frequency
r = np.corrcoef(frequency, unique_products)[0, 1]
print(f'\nCorrelation with frequency: r = {r:.3f}')
print('(High correlation expected: more trips usually means more product variety)')

In [ ]:
# VERIFY example customer
ex_up = unique_products.loc[EXAMPLE_CID]
manual = h_txns['StockCode'].nunique()
print(f'Customer {EXAMPLE_CID}  unique_products = {ex_up}  (manual = {manual})')
assert ex_up == manual

### Does high correlation mean we should drop unique_products?

No. Correlated predictors are not a problem to eliminate — they are the reason
regularization matters. Lasso may zero out `unique_products` if `frequency` explains
the same variance. Ridge will shrink both. We include it and let the model decide.

This is the core of the Step 6 regularization comparison.

---

## Section 10 — Tenure

### CONCEPT

**Tenure** measures the span of a customer's historical activity:
```
tenure_days = last_historical_purchase − first_historical_purchase  [days]
```

A customer with tenure = 500 days has been shopping over a 500-day period.
A customer with tenure = 0 made all their purchases on the same day.

Tenure captures **customer longevity** — a long-standing customer may behave
differently from a new one even if their total spend is similar.

### Formula

```
tenure_days = max(InvoiceDate) − min(InvoiceDate)  [historical window only]
```

In [ ]:
# STEP 1 — find first and last purchase dates per customer
first_purchase = (
    historical_df
    .groupby('Customer ID')['InvoiceDate']
    .min()
    .rename('first_purchase')
)

# We already have last_purchase from Section 5
# But let's recalculate here for clarity
last_purchase_s5 = (
    historical_df
    .groupby('Customer ID')['InvoiceDate']
    .max()
    .rename('last_purchase')
)

# STEP 2 — difference in days
tenure = (last_purchase_s5 - first_purchase).dt.days.rename('tenure_days')

print('tenure_days (first 5):')
print(tenure.head())

In [ ]:
# INSPECT — and the critical edge case
print('Tenure distribution (all historical customers):')
for label, val in [('Min', tenure.min()), ('Median', tenure.median()),
                   ('Mean', tenure.mean()), ('Max', tenure.max())]:
    print(f'  {label:<8}: {val:.1f} days')
print()
print(f'tenure == 0: {(tenure==0).sum():,} customers  ({(tenure==0).mean()*100:.1f}%)')
print()
print('These customers made ALL their purchases on a single calendar day.')

### Edge case: tenure == 0

**37.9% of customers** have `tenure_days = 0`.

This is NOT a data error. It means one of:
1. The customer made exactly one purchase (single invoice)
2. The customer placed multiple orders but all on the same day

This is real customer behaviour — one-time buyers or single-day shoppers.
It matters because `purchase_rate = frequency / tenure_days` would divide by zero
for these customers. We handle this explicitly in Section 11.

In [ ]:
# Show examples of tenure==0 customers
zero_tenure_ids = tenure[tenure == 0].index[:3].tolist()
print('Examples of tenure=0 customers:')
for cid in zero_tenure_ids:
    rows = historical_df[historical_df['Customer ID'] == cid][['Invoice','InvoiceDate','line_total']]
    f = rows['Invoice'].nunique()
    print(f'\nCustomer {cid}: {f} invoice(s), all on same day:')
    print(rows.to_string(index=False))

In [ ]:
# VERIFY example customer
ex_tenure = tenure.loc[EXAMPLE_CID]
manual    = (h_txns['InvoiceDate'].max() - h_txns['InvoiceDate'].min()).days
print(f'Customer {EXAMPLE_CID}  tenure_days = {ex_tenure}  (manual = {manual})')
assert ex_tenure == manual

---

## Section 11 — Purchase Rate

### CONCEPT

**Purchase rate** attempts to measure how frequently a customer shops within their
active period:
```
purchase_rate = frequency / tenure_days
```

Interpretation: orders per day. A purchase_rate of 0.05 means roughly one order
every 20 days. A rate of 0.01 means one order every 100 days.

### The critical problem: tenure == 0

37.9% of customers have `tenure_days = 0`. For them:
```
purchase_rate = frequency / 0  →  ZeroDivisionError (or NaN/Inf)
```

We must decide what to assign these customers. This is not a trivial choice.

### Two candidate definitions

In [ ]:
# OPTION A: frequency / max(tenure_days, 1)
# Treats tenure=0 as if it were 1 day
purchase_rate_A = (frequency / tenure.clip(lower=1)).rename('purchase_rate_A')

# OPTION B: 0.0 when tenure==0; frequency/tenure otherwise
import numpy as np
purchase_rate_B = pd.Series(
    np.where(tenure > 0, frequency / tenure, 0.0),
    index=tenure.index,
    name='purchase_rate_B'
)

# Compare for tenure==0 customers
zero_ten = tenure[tenure == 0].index
print('Customers with tenure=0:')
print(f'  Count             : {len(zero_ten):,}')
print(f'  Avg frequency     : {frequency.loc[zero_ten].mean():.2f}')
print()
print('Resulting purchase_rate values:')
print(f'  Option A (freq/max(tenure,1)) median : {purchase_rate_A.loc[zero_ten].median():.4f}')
print(f'  Option B (assign 0.0)         median : {purchase_rate_B.loc[zero_ten].median():.4f}')
print()
print('For customers with tenure>0:')
print(f'  purchase_rate median: {purchase_rate_B[tenure>0].median():.4f}')
print(f'  purchase_rate max   : {purchase_rate_B[tenure>0].max():.4f}')

### Analysis and Decision

**Option A — `frequency / max(tenure_days, 1)`**

Business interpretation: "A one-day shopper who placed 5 orders in that day has purchase_rate = 5."
That is **5 purchases per day** — far higher than the most active long-term customer.
This is mathematically defensible but can be misleading: it suggests the customer is
extremely active when in reality we simply cannot measure their cadence from one day.

A customer with `frequency=3, tenure=0` gets `purchase_rate_A = 3.0`.
A customer with `frequency=30, tenure=300` gets `purchase_rate = 0.1`.
Option A makes the one-day shopper appear 30× more active. Is that what we want?

**Option B — `0.0` when `tenure_days == 0`**

Business interpretation: "We cannot measure the purchasing cadence of a one-day customer.
We assign the neutral value 0, not because they are inactive, but because we lack data."
The `frequency`, `monetary`, and `recency` features already capture what they bought.
`purchase_rate` adds nothing new for a one-day customer beyond what `frequency` already encodes.

**Decision: Option B.**
- Avoids inflating purchase_rate for one-day customers
- Does not introduce artificial variance
- `frequency` already captures the one-day customer's order count
- 0.0 is interpretable: "unknown cadence" at the neutral baseline

In [ ]:
# CHOSEN IMPLEMENTATION — Option B
purchase_rate = purchase_rate_B.rename('purchase_rate')

# Verify: no NaN, no Inf
assert purchase_rate.isna().sum() == 0,      'NaN found in purchase_rate'
assert np.isinf(purchase_rate).sum() == 0,   'Inf found in purchase_rate'

print('purchase_rate statistics (tenure>0 customers only):')
pos = purchase_rate[tenure > 0]
for label, val in [('Median', pos.median()), ('Mean', pos.mean()), ('Max', pos.max())]:
    print(f'  {label:<8}: {val:.4f}')
print()
print(f'Customers with purchase_rate=0 (tenure==0): {(purchase_rate==0).sum():,}')

In [ ]:
# VERIFY example customer
ex_pr = purchase_rate.loc[EXAMPLE_CID]
manual = frequency.loc[EXAMPLE_CID] / tenure.loc[EXAMPLE_CID]
print(f'Customer {EXAMPLE_CID}  purchase_rate = {ex_pr:.6f}')
print(f'Manual check: {frequency.loc[EXAMPLE_CID]} / {tenure.loc[EXAMPLE_CID]} = {manual:.6f}')
assert abs(ex_pr - manual) < 1e-9

---

## Section 12 — Country

### CONCEPT

Country identifies the geographic location associated with a customer.
It is a **categorical** feature — we store the raw string here.
Encoding (one-hot or ordinal) belongs to the preprocessing pipeline in Step 6.

### Questions to answer first

1. How many countries appear? How dominant is the UK?
2. Do any customers have transactions from multiple countries?
3. If yes, which country do we assign as the customer-level country?

In [ ]:
# Distribution of countries (using most-recent-country per customer)
cust_country_recent = (
    historical_df
    .sort_values('InvoiceDate')
    .groupby('Customer ID')['Country']
    .last()
    .rename('country')
)

cc = cust_country_recent.value_counts()
uk_pct = cc.get('United Kingdom', 0) / len(cust_country_recent) * 100
print('Customers per country (top 15):')
print(cc.head(15).to_string())
print()
print(f'UK customers    : {cc.get("United Kingdom",0):,} ({uk_pct:.1f}%)')
print(f'Non-UK customers: {(cust_country_recent!="United Kingdom").sum():,} ({100-uk_pct:.1f}%)')
print(f'Total countries : {len(cc)}')
print(f'Countries with only 1 customer: {(cc==1).sum()}')

In [ ]:
# Multi-country customers
countries_per_cust = (
    historical_df
    .groupby('Customer ID')['Country']
    .nunique()
    .rename('n_countries')
)
multi = countries_per_cust[countries_per_cust > 1]
print(f'Customers with transactions in more than 1 country: {len(multi):,}')

if len(multi) > 0:
    print()
    for cid in multi.index[:3]:
        rows = historical_df[historical_df['Customer ID'] == cid][
            ['InvoiceDate', 'Country']
        ].sort_values('InvoiceDate')
        print(f'Customer {cid} — countries over time:')
        print(rows.drop_duplicates().to_string(index=False))
        print()

### Decision: Most-recent-country

Two options:

| Option | Logic | Advantage | Disadvantage |
|---|---|---|---|
| Most-recent | Country of the chronologically last transaction | Reflects current geography | Can change year to year |
| Most-frequent | Country appearing in most transactions | Stable; represents habitual location | May miss recent relocation |

**Decision: most-recent-country.**

Reasoning:
1. Only 6 customers appear in >1 country — the choice barely matters in practice
2. The RFM framework prioritises recency everywhere else — consistency supports recency here too
3. If a customer's last known location is Portugal, Portugal is the most relevant geographic signal

**Implementation note:** The sort must happen BEFORE the groupby `.last()` call.
Without sorting by `InvoiceDate`, `.last()` returns the last row in DataFrame order —
which is arbitrary and may not be the most recent purchase.

In [ ]:
# CORRECT implementation — sort first, then take .last()
country = (
    historical_df
    .sort_values('InvoiceDate')           # ← critical: must sort before .last()
    .groupby('Customer ID')['Country']
    .last()
    .rename('country')
)

# Verify example customer
ex_country = country.loc[EXAMPLE_CID]
manual = h_txns.sort_values('InvoiceDate').iloc[-1]['Country']
print(f'Customer {EXAMPLE_CID}  country = {ex_country}  (manual = {manual})')
assert ex_country == manual

---

## Section 13 — Assemble X: The Customer Feature Matrix

Now we combine all manually computed features into a single customer-level DataFrame.

In [ ]:
# Combine all manual features into one DataFrame
X_manual = pd.concat([
    recency.rename('recency_days'),
    frequency,
    monetary,
    avg_basket_value,
    unique_products,
    tenure,
    purchase_rate,
    country,
], axis=1)

print('X_manual shape:', X_manual.shape)
print()
print('Columns:', list(X_manual.columns))
print()
print('Head:')
print(X_manual.head(5).to_string())

In [ ]:
# Validate structure
print('=== X VALIDATION ===')
print(f'Rows              : {len(X_manual):,}')
print(f'Unique customers  : {X_manual.index.nunique():,}')
print(f'Duplicate rows    : {X_manual.index.duplicated().sum()}')
print(f'Missing values    :')
print(X_manual.isnull().sum().to_string())
assert X_manual.index.duplicated().sum() == 0, 'Duplicate Customer IDs found!'
assert X_manual.isnull().sum().sum() == 0, 'Missing values found!'
print()
print('One row per customer: CONFIRMED')
print('No missing values   : CONFIRMED')

### Why Customer ID must NOT be used as a predictor

`Customer ID` is a **database key** — an arbitrary integer assigned during system registration.
It has no mathematical relationship with future spending.
If we pass it as a numeric feature, the model might learn spurious correlations
(e.g. "customers with ID > 15000 spend more") that are coincidental in the training data
and will not generalise.

`Customer ID` stays in the DataFrame as a **join key** only.
When we pass the data to the regression model, we will explicitly exclude it.

---

## Section 14 — Build y: Future Revenue (Separately from X)

**Important:** From this point on, we switch to `future_df`.
We do NOT touch `historical_df` in this section.

### CONCEPT

```
future_revenue = SUM(line_total)  for InvoiceDate >= prediction_date
```

For customers with no post-cutoff transactions: `future_revenue = 0`

### Why is zero a genuine observation, not missing data?

A zero means: "This customer was active historically and we observed them for 99 days
after the cutoff — they simply did not purchase during that window."

This is not a data gap. It is the actual answer to "how much will they spend?"
Removing or imputing these zeros would bias the model toward active customers
and make it unable to predict churn or low-value outcomes.

In [ ]:
# STEP 1 — compute future revenue from future_df ONLY
future_revenue_raw = (
    future_df                               # <- POST-cutoff only
    .groupby('Customer ID')['line_total']
    .sum()
    .rename('future_revenue')
)

print(f'Customers with positive future revenue: {len(future_revenue_raw):,}')
print(f'Their total future revenue: £{future_revenue_raw.sum():,.2f}')
print()
print('These are Group C customers (historical + future)')

In [ ]:
# STEP 2 — left-join onto historical customers; fill NaN → 0
# The index of X_manual contains all historical customers
future_revenue = (
    X_manual.index.to_series()                   # all historical customer IDs
    .to_frame(name='Customer ID')
    .merge(
        future_revenue_raw.reset_index(),
        on='Customer ID',
        how='left'                               # keep ALL historical customers
    )
    .set_index('Customer ID')['future_revenue']
    .fillna(0.0)                                # Group A: no future transactions → 0
)

zero_pct = (future_revenue == 0).mean() * 100
pos_pct  = (future_revenue > 0).mean()  * 100

print('Future revenue distribution (historical customers):')
print(f'  Group A (zero rev) : {(future_revenue==0).sum():,}  ({zero_pct:.1f}%)')
print(f'  Group C (pos rev)  : {(future_revenue>0).sum():,}  ({pos_pct:.1f}%)')
print(f'  Mean               : £{future_revenue.mean():,.2f}')
print(f'  Median             : £{future_revenue.median():,.2f}')
print(f'  Max                : £{future_revenue.max():,.2f}')

In [ ]:
# Why are future-only customers excluded?
future_only = set(future_df['Customer ID'].unique()) - set(X_manual.index)
hist_only   = set(X_manual.index) - set(future_df['Customer ID'].unique())
print(f'Future-only customers (Group B) : {len(future_only):,}')
print(f'  These customers have y but no X — cannot build a feature vector.')
print(f'  Excluded from the modelling population.')
print()
print(f'Historical-only (Group A)       : {len(hist_only):,}')
print(f'  These customers have X but y=0 — kept with future_revenue=0.')

In [ ]:
# VERIFY example customer
ex_frev = future_revenue.loc[EXAMPLE_CID]
manual  = f_txns['line_total'].sum()
print(f'Customer {EXAMPLE_CID}  future_revenue = £{ex_frev:.2f}  (manual = £{manual:.2f})')
assert abs(ex_frev - manual) < 0.01

---

## Section 15 — Manual End-to-End: Customer 12374

Let's trace every calculation from raw transactions to a complete modelling row.
You should be able to reproduce every number with a basic calculator.

In [ ]:
print('=' * 60)
print(f'CUSTOMER {EXAMPLE_CID} — COMPLETE FEATURE DERIVATION')
print('=' * 60)
print()

# Step 1: show historical transactions
print('HISTORICAL TRANSACTIONS:')
print(h_txns[['Invoice','InvoiceDate','StockCode','Quantity','Price','line_total']].to_string(index=False))
print()

# Step 2: manually calculate each feature
h_last   = h_txns['InvoiceDate'].max()
h_first  = h_txns['InvoiceDate'].min()
h_inv_n  = h_txns['Invoice'].nunique()
h_mon    = h_txns['line_total'].sum()
h_inv_sp = h_txns.groupby('Invoice')['line_total'].sum()
h_aov    = h_inv_sp.mean()
h_up     = h_txns['StockCode'].nunique()
h_ten    = (h_last - h_first).days
h_pr     = h_inv_n / h_ten if h_ten > 0 else 0.0
h_co     = h_txns.sort_values('InvoiceDate').iloc[-1]['Country']
h_rec    = (CUTOFF - h_last).days

print(f'PREDICTION DATE : {CUTOFF.date()}')
print()
print(f'recency_days     = {CUTOFF.date()} - {h_last.date()} = {h_rec} days')
print(f'frequency        = {h_inv_n} distinct invoices  (NOT {len(h_txns)} rows)')
print(f'monetary         = £{h_mon:.2f}')
print(f'avg_basket_value = mean({list(h_inv_sp.values)}) = £{h_aov:.3f}')
print(f'unique_products  = {h_up} distinct StockCodes')
print(f'tenure_days      = {h_last.date()} - {h_first.date()} = {h_ten} days')
print(f'purchase_rate    = {h_inv_n} / {h_ten} = {h_pr:.6f} orders/day')
print(f'country          = {h_co}  (most recent invoice)')
print()
print('FUTURE TRANSACTIONS:')
print(f_txns[['Invoice','InvoiceDate','StockCode','Quantity','Price','line_total']].to_string(index=False))
print()
f_rev = f_txns['line_total'].sum()
print(f'future_revenue       = £{f_rev:.2f}')
print(f'log1p_future_revenue = log({f_rev:.2f} + 1) = log({f_rev+1:.2f}) = {math.log(f_rev+1):.6f}')

In [ ]:
# Show this customer's row in the final dataset
customer_row = pd.DataFrame({
    'Customer ID':         [EXAMPLE_CID],
    'recency_days':        [h_rec],
    'frequency':           [h_inv_n],
    'monetary':            [h_mon],
    'avg_basket_value':    [h_aov],
    'unique_products':     [h_up],
    'tenure_days':         [h_ten],
    'purchase_rate':       [h_pr],
    'country':             [h_co],
    'future_revenue':      [f_rev],
    'log1p_future_revenue':[math.log(f_rev + 1)],
})

print('FINAL MODELLING ROW for Customer 12374:')
print(customer_row.to_string(index=False))

---

## Section 16 — Target Distribution: To Transform or Not?

### Analyse raw future_revenue

In [ ]:
print('Future Revenue distribution (all historical customers):')
for label, val in [('Zero (=0)', f'{(future_revenue==0).sum():,}  ({(future_revenue==0).mean()*100:.1f}%)'),
                   ('Positive (>0)', f'{(future_revenue>0).sum():,}  ({(future_revenue>0).mean()*100:.1f}%)'),
                   ('Mean', f'£{future_revenue.mean():,.2f}'),
                   ('Median', f'£{future_revenue.median():,.2f}'),
                   ('75th %', f'£{future_revenue.quantile(0.75):,.2f}'),
                   ('99th %', f'£{future_revenue.quantile(0.99):,.2f}'),
                   ('Max', f'£{future_revenue.max():,.2f}')]:
    print(f'  {label:<15}: {val}')
print(f'  Skew          : {future_revenue.skew():.2f}  <- extreme right skew')

In [ ]:
# Compare raw vs log1p distributions
log1p_rev = np.log1p(future_revenue)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw — show positive values only for clarity
pos = future_revenue[future_revenue > 0]
cap = pos.quantile(0.99)
axes[0].hist(pos[pos <= cap], bins=60, color='salmon', edgecolor='white')
axes[0].set_xlabel('Future Revenue (£), positive only')
axes[0].set_ylabel('Customers')
zero_pct2 = (future_revenue == 0).mean() * 100
axes[0].set_title(f'Raw Future Revenue\n(positive only; {zero_pct2:.0f}% of customers are zero)')
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'£{x:,.0f}'))

# log1p — include all customers (zeros map to 0)
axes[1].hist(log1p_rev, bins=60, color='steelblue', edgecolor='white')
axes[1].set_xlabel('log1p(Future Revenue)')
axes[1].set_ylabel('Customers')
axes[1].set_title(f'log1p(Future Revenue)\n(all customers; log1p(0)=0)')

plt.tight_layout()
plt.show()

print(f'Raw skew   : {future_revenue.skew():.2f}')
print(f'log1p skew : {log1p_rev.skew():.2f}')

### Why log1p may help linear regression

| Aspect | Raw future_revenue | log1p(future_revenue) |
|---|---|---|
| Skew | 16.27 (extreme) | 1.03 (near-symmetric) |
| Zero handling | Valid zero | log1p(0) = 0 (safe) |
| Error unit | Absolute £ | Proportional (log-£) |
| OLS assumption | Violated (non-normal residuals) | Better satisfied |
| Outlier influence | A few big spenders dominate loss | Loss spread more evenly |

**Why log1p(0) = 0 is safe:** The standard `log(x)` function is undefined at zero (it's -∞).
`log1p(x) = log(x+1)` is defined at zero: `log1p(0) = log(1) = 0`.
This maps our "no future revenue" customers cleanly to 0 in the log space.

### Why NOT to blindly choose log1p

When you train on log1p(y), your model predicts log-revenue, not £-revenue.
To recover £-predictions you apply `expm1(prediction)`.

However: `E[expm1(predicted_log_revenue)] ≠ E[actual revenue]`

The expected value of a log-normal is not the exponentiated mean of the log.
There is a **Jensen's inequality correction**: `E[Y] = exp(μ + σ²/2)`.
If your business metric is average predicted revenue, you need this correction.

### Decision

We use `log1p_future_revenue` as the **primary regression target** because:
1. The raw skew of 16.27 makes OLS numerically unstable and biased toward outliers
2. log1p compression (skew 1.03) satisfies homoscedasticity better
3. Zero maps safely to 0
4. We preserve `future_revenue` (raw £) for final evaluation metrics

In [ ]:
# Create log1p target — both are preserved
log1p_future_revenue = np.log1p(future_revenue).rename('log1p_future_revenue')

print('log1p(future_revenue) — first 5 customers:')
print(pd.DataFrame({'future_revenue': future_revenue.head(),
                    'log1p_future_revenue': log1p_future_revenue.head()}).to_string())

---

## Section 17 — Leakage Audit

The table below audits every candidate variable for temporal leakage risk.

| Variable | Source period | Allowed as X? | Reason |
|---|---|---|---|
| `recency_days` | Historical (< cutoff) | ✅ YES | Uses only pre-cutoff dates |
| `frequency` | Historical (< cutoff) | ✅ YES | Counts pre-cutoff invoices only |
| `monetary` | Historical (< cutoff) | ✅ YES | Sums pre-cutoff line_total only |
| `avg_basket_value` | Historical (< cutoff) | ✅ YES | Derived from pre-cutoff invoices |
| `unique_products` | Historical (< cutoff) | ✅ YES | Counts pre-cutoff StockCodes |
| `tenure_days` | Historical (< cutoff) | ✅ YES | Uses pre-cutoff min/max dates |
| `purchase_rate` | Historical (< cutoff) | ✅ YES | Derived from frequency and tenure |
| `country` | Historical (< cutoff) | ✅ YES | Most-recent pre-cutoff Country |
| `future_revenue` | Future (>= cutoff) | ❌ NO (is the TARGET y) | Using it as a feature means X contains y |
| `returned_in_future` | Future (>= cutoff) | ❌ NO (leakage) | Tells model who returned; created from future data |
| `future_transaction_count` | Future (>= cutoff) | ❌ NO (leakage) | Number of post-cutoff invoices — derived from future |
| `Customer ID` | Not time-indexed | ❌ NO (not predictive) | Arbitrary system key — no causal relationship with spend |

### Leakage tests (automated)

In [ ]:
# TEST 1: X features must not change when a post-cutoff row is added
pre_test   = historical_df.copy()
fut_test   = future_df.copy()
LEAK_CID   = X_manual.index[0]

rec_before = recency.loc[LEAK_CID]
mon_before = monetary.loc[LEAK_CID]

# Confirm: features were built from historical_df (no future data)
# A post-cutoff transaction for LEAK_CID should NOT change recency or monetary
print('Leakage test: X must only use historical data.')
print(f'Customer {LEAK_CID} recency before = {rec_before}')
print(f'Customer {LEAK_CID} monetary before = £{mon_before:.2f}')
print()

# Rebuild features from scratch — with future_df included in the input
df_with_leak = pd.concat([historical_df, future_df], ignore_index=True)
hist_leak = df_with_leak[df_with_leak['InvoiceDate'] < CUTOFF]  # filter pre-cutoff
rec_after = (CUTOFF - hist_leak.groupby('Customer ID')['InvoiceDate'].max()).dt.days
mon_after = hist_leak.groupby('Customer ID')['line_total'].sum()

assert rec_after.loc[LEAK_CID] == rec_before, 'LEAK: recency changed!'
assert abs(mon_after.loc[LEAK_CID] - mon_before) < 0.01, 'LEAK: monetary changed!'
print('TEST 1 PASSED: X features unchanged when future data added to raw DataFrame')
print('(because we always filter InvoiceDate < CUTOFF before computing features)')

In [ ]:
# TEST 2: future_revenue must not change when historical transaction is inflated
fut_rev_before = future_revenue.loc[EXAMPLE_CID]

# Inflate historical monetary — should NOT affect future_revenue
df_inflated = historical_df.copy()
df_inflated.loc[df_inflated['Customer ID'] == EXAMPLE_CID, 'line_total'] = 999999.0
# Recompute future_revenue from original future_df (unchanged)
fut_rev_check = future_df.groupby('Customer ID')['line_total'].sum()
fut_after = fut_rev_check.get(EXAMPLE_CID, 0.0)

assert abs(fut_after - fut_rev_before) < 0.01, 'LEAK: future_revenue changed from historical edit!'
print('TEST 2 PASSED: future_revenue unaffected by changes to historical data')
print(f'  fut_rev before: £{fut_rev_before:.2f}  |  fut_rev after: £{fut_after:.2f}')

---

## Section 18 — Compare Manual Implementation with Production src/ Code

The project already has a production feature-engineering module at
`src/features/build_features.py`. Now we verify that it produces identical results
to our manual notebook implementation.

### What the production functions do

**`build_rfm_features(df, prediction_date)`**
- INPUT: cleaned transaction DataFrame + cutoff date
- OUTPUT: one row per historical customer with all X features
- Internally: filters to `InvoiceDate < prediction_date`, sorts by date, then
  groups and aggregates in a single `.agg()` call plus derived columns

**`build_target(df, prediction_date)`**
- INPUT: cleaned transaction DataFrame + cutoff date
- OUTPUT: one row per post-cutoff customer with `future_revenue` and `log1p_future_revenue`
- Internally: filters to `InvoiceDate >= prediction_date` only

**`build_customer_dataset(df, prediction_date)`**
- INPUT: cleaned transaction DataFrame + cutoff date
- OUTPUT: modelling-ready dataset (X + y joined on Customer ID)
- Calls both of the above, then left-joins on historical customers

In [ ]:
# Call the production functions
src_features = build_rfm_features(df, PREDICTION_DATE)
src_target   = build_target(df, PREDICTION_DATE)
src_dataset  = build_customer_dataset(df, PREDICTION_DATE)

print('Production function outputs:')
print(f'  build_rfm_features  -> {src_features.shape}')
print(f'  build_target        -> {src_target.shape}')
print(f'  build_customer_dataset -> {src_dataset.shape}')

In [ ]:
# Compare manual vs production feature values for the example customer
src_row = src_features[src_features['Customer ID'] == EXAMPLE_CID].iloc[0]

print(f'Comparison for Customer {EXAMPLE_CID}:')
print(f'{'Feature':<22} {'Manual':>12} {'Production':>12} {'Match':>8}')
print('-' * 60)

comparisons = [
    ('recency_days',      recency.loc[EXAMPLE_CID],          src_row['recency_days']),
    ('frequency',         frequency.loc[EXAMPLE_CID],         src_row['frequency']),
    ('monetary',          monetary.loc[EXAMPLE_CID],          src_row['monetary']),
    ('avg_basket_value',  avg_basket_value.loc[EXAMPLE_CID],  src_row['avg_basket_value']),
    ('unique_products',   unique_products.loc[EXAMPLE_CID],   src_row['unique_products']),
    ('tenure_days',       tenure.loc[EXAMPLE_CID],            src_row['tenure_days']),
    ('purchase_rate',     purchase_rate.loc[EXAMPLE_CID],     src_row['purchase_rate']),
    ('country',           country.loc[EXAMPLE_CID],           src_row['country']),
]

for feat, manual_val, src_val in comparisons:
    if isinstance(manual_val, str):
        match = 'YES' if manual_val == src_val else 'NO'
        print(f'{feat:<22} {str(manual_val):>12} {str(src_val):>12} {match:>8}')
    else:
        match = 'YES' if abs(float(manual_val) - float(src_val)) < 1e-6 else 'NO'
        print(f'{feat:<22} {float(manual_val):>12.4f} {float(src_val):>12.4f} {match:>8}')

In [ ]:
# Full population comparison — all customers, all numeric features
manual_numeric = X_manual.drop(columns=['country']).sort_index()
src_numeric    = src_features.set_index('Customer ID').drop(columns=['country']).sort_index()

# Align columns (production may have different column order)
shared_cols = [c for c in manual_numeric.columns if c in src_numeric.columns]
diff = (manual_numeric[shared_cols] - src_numeric[shared_cols]).abs()

print('Max absolute difference per feature (manual vs production):')
print(diff.max().to_string())
print()
max_diff = diff.max().max()
print(f'Maximum difference across all customers and features: {max_diff:.2e}')
if max_diff < 1e-6:
    print('CONFIRMED: manual implementation == production implementation (within floating-point precision)')
else:
    print('WARNING: differences found — investigate before proceeding')

### What we confirmed

The production `build_rfm_features` function produces values identical to the
step-by-step manual implementation in this notebook.

The one code change made during this step: `pre = pre.sort_values('InvoiceDate')`
was added to guarantee the `country` assignment uses chronological order.
Without this sort, `.agg(..., "last")` returns the last row in DataFrame insertion
order — which is arbitrary. The fix ensures the most-recent-InvoiceDate country
is always selected for multi-country customers.

---

## Section 19 — Final Modelling Dataset

We now assemble the complete customer-level dataset using the production functions.

In [ ]:
# Use the production function — it reuses build_rfm_features and build_target
customer_df = build_customer_dataset(df, PREDICTION_DATE)

print('Final modelling dataset:')
print(f'  Shape   : {customer_df.shape}')
print(f'  Columns : {list(customer_df.columns)}')
print()
customer_df.head(5)

In [ ]:
# Validate the final dataset
print('=== DATASET VALIDATION ===')
print(f'Total rows             : {len(customer_df):,}')
print(f'Unique Customer IDs    : {customer_df["Customer ID"].nunique():,}')
print(f'Duplicate IDs          : {customer_df["Customer ID"].duplicated().sum()}')
print()
print('Missing values per column:')
print(customer_df.isnull().sum().to_string())
print()
print(f'Group A (future_revenue=0): {(customer_df["future_revenue"]==0).sum():,}')
print(f'Group C (future_revenue>0): {(customer_df["future_revenue"]>0).sum():,}')
assert customer_df['Customer ID'].duplicated().sum() == 0
assert customer_df.isnull().sum().sum() == 0
print()
print('All checks passed.')

In [ ]:
# Show numeric summary
customer_df.describe()

### What is NOT in this dataset

We deliberately excluded:
- **StandardScaler** — belongs to the sklearn preprocessing pipeline in Step 6
- **OneHotEncoder** for country — belongs to the preprocessing pipeline in Step 6
- **train_test_split** — belongs to Step 6 model evaluation
- **log1p of features** — features are stored as raw values; the model pipeline handles scaling
- **Customer ID as a model input** — it is present as a join key but excluded from the feature matrix

This dataset is the clean, unprocessed modelling input.
All transformations applied inside the regression pipeline maintain correctness
during cross-validation (no leakage from preprocessing).

---

## Section 20 — Save to data/processed/

### Data directory structure

```
data/
├── raw/           ← IMMUTABLE — original Excel file, never modified
├── interim/       ← cleaned transaction-level data (transactions_cleaned.csv)
└── processed/     ← customer-level modelling data (customer_dataset.csv)  ← HERE
```

Each tier has a specific purpose:
- `raw/` — source of truth; never touch it
- `interim/` — output of the cleaning pipeline (Step 2)
- `processed/` — output of feature engineering (this step)

In [ ]:
# Save using the production function (path comes from project config)
save_customer_dataset(customer_df, CUSTOMER_DATASET_CSV)

print()
print(f'Saved to: {CUSTOMER_DATASET_CSV}')
print(f'Rows    : {len(customer_df):,}')
print(f'Columns : {list(customer_df.columns)}')

In [ ]:
# Verify the file exists and can be read back
reloaded = pd.read_csv(CUSTOMER_DATASET_CSV)
assert len(reloaded) == len(customer_df), 'Row count mismatch after save/reload!'
print(f'Reload check: {len(reloaded):,} rows — OK')
print(f'Columns: {list(reloaded.columns)}')

---

## Section 21 — Unit Tests

The production tests live in `tests/test_features.py` and were expanded during this step.
Let's run them and confirm all pass.

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '-m', 'pytest', 'tests/test_features.py', '-v', '--tb=short'],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT)
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

### What each test group protects

| Group | Tests | What it catches |
|---|---|---|
| Structural | 8 | Wrong column names, wrong number of rows, missing customers |
| Exact values | 13 | Wrong formulas (e.g., using row count instead of invoice count) |
| Edge cases | 7 | tenure=0 NaN/Inf, multi-country wrong selection, single-invoice behaviour |
| build_target | 6 | Future revenue computed from wrong window |
| build_customer_dataset | 11 | Join errors, NaN instead of 0 for Group A, log1p consistency |
| Leakage | 6 | Post-cutoff data in X, pre-cutoff data in y, target in features |

**Key leakage tests:**
- `test_leakage_no_future_monetary_in_features` — customer 2002 monetary must be 32.00 not 44.00
- `test_leakage_x_uses_only_historical_transactions` — adding a future row does not change X
- `test_leakage_y_uses_only_future_transactions` — modifying historical data does not change y

---

## Final Summary

### WHAT WE BUILT

A **customer-level modelling dataset** with 4,374 rows and 11 columns, saved to
`data/processed/customer_dataset.csv`.

Each row represents one historical customer, described by 8 features derived from
their pre-cutoff transactions, and labelled with their post-cutoff revenue.

### WHAT I LEARNED

1. **Temporal splitting** is the foundational principle: X from before the cutoff, y from after.
2. **frequency ≠ row count** — always count distinct invoices, not transaction rows.
3. **monetary ≠ future_revenue** — they measure the same type of value but in different time windows.
4. **avg_basket_value = monetary / frequency** (approximately) — including all three in OLS creates multicollinearity.
5. **tenure == 0 affects 37.9% of customers** — must be handled explicitly before computing purchase_rate.
6. **Country requires sorting** before `.last()` to guarantee chronological order.
7. **Zero future_revenue is genuine** — it means the customer did not return, not that data is missing.
8. **log1p compresses skew from 16.27 to 1.03** — making OLS assumptions more reasonable.
9. **Group B customers (future-only) are excluded** — they have y but no X.
10. **Leakage tests** should be part of every feature engineering pipeline.

### FINAL FEATURES

| Feature | Formula | Source window |
|---|---|---|
| `recency_days` | cutoff − max(InvoiceDate) | Historical |
| `frequency` | COUNT(DISTINCT Invoice) | Historical |
| `monetary` | SUM(line_total) | Historical |
| `avg_basket_value` | MEAN(SUM(line_total) per invoice) | Historical |
| `unique_products` | COUNT(DISTINCT StockCode) | Historical |
| `tenure_days` | max(InvoiceDate) − min(InvoiceDate) | Historical |
| `purchase_rate` | frequency/tenure if tenure>0 else 0 | Historical |
| `country` | Country of most-recent invoice | Historical |

### TARGET DEFINITION

**Primary:** `log1p_future_revenue = log(future_revenue + 1)`
**Preserved:** `future_revenue` (raw £) for evaluation

### FORECAST HORIZON

**99 days** (2011-09-01 to 2011-12-09) — confirmed from actual data, not assumed.

### EDGE CASES HANDLED

| Edge case | Population | Decision |
|---|---|---|
| tenure_days == 0 | 37.9% of customers | purchase_rate = 0.0 |
| Group B (future-only) | 472 customers | Excluded |
| Multi-country customers | 6 customers | Assign most-recent InvoiceDate country |
| future_revenue = 0 | 64.7% of customers | Genuine zero, preserved |

### LEAKAGE CHECKS

All 6 automated leakage tests pass. X strictly from pre-cutoff; y strictly from post-cutoff.

### PRODUCTION CODE USED

- `build_rfm_features` — reused with one fix (added `pre.sort_values('InvoiceDate')`)
- `build_target` — reused without change
- `build_customer_dataset` — reused without change
- `save_customer_dataset` — reused without change

### TESTS EXECUTED

**96/96 passed** (51 feature tests, 27 data tests, 18 model tests)

### NEXT STEP

**Step 6 — Regression Modelling:** Train and compare OLS, Ridge, Lasso, and ElasticNet on
the `customer_dataset.csv` produced here. The multicollinearity between features
(frequency ↔ monetary, frequency ↔ unique_products, monetary ↔ avg_basket_value, tenure ↔ frequency)
makes regularization essential — this is exactly what Step 6 will demonstrate.